# Module 10: Important Modules

**Utrains Python Fundamentals** &middot; lab notebook

*Import the standard library, set up a virtual environment, and meet openai, langchain, and langgraph.*

## By the end of this notebook you can

- Say what a module is and what import actually does
- Use all three import patterns
- Measure time and read the operating system with time and os
- Keep secrets out of your code and out of source control
- Say what openai, langchain and langgraph each do, and when to pick which

## How to work through it

It follows the Module 10 slide deck, slide by slide.

The headings below are the slide numbers from the deck. The explanation for
each one is on the slide and in the [README](../README.md); this notebook is
where you run the code.

Run every cell in order with **Shift + Enter**.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

**Assumed knowledge.** Modules 1 to 9. The AI package slides need network access and an API key, so their code is shown as reference and paired with a standard library stand-in you can actually run.

## Slide 2 &middot; What Is a Module?

In [ ]:
import time            # standard library, comes with Python

# import ollama       # third party, pip install ollama first

print("time module loaded:", time.__name__)

## Slide 3 &middot; Import Patterns

In [ ]:
import datetime as dt
from math import sqrt

print(dt.date.today())
print(sqrt(16))

## Slide 4 &middot; Measuring Time

In [ ]:
import time

t0 = time.time()
time.sleep(0.3)
print(f"Waited {time.time() - t0:.1f}s")

---

### Your turn 1

Measure how long a fake health check takes and print the answer to two decimal places. Two ideas here: the import statement, and the call that reads the clock.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
# TODO: bring in the module that can pause and read the clock.
____ time


def run_healthcheck():
    time.sleep(0.25)
    return "healthy"


# TODO: read the clock before and after.
start = time.____()
status = run_healthcheck()
elapsed = time.____() - start

print(f"health check returned {status} in {elapsed:.2f}s")

## Slide 5 &middot; Working with the Operating System

In [ ]:
import os

print(os.getcwd())                      # current directory
print(sorted(os.listdir("."))[:10])     # files here

## Slide 6 &middot; Config Values versus Secrets

In [ ]:
import os

MODEL = "claude-sonnet-4-6"
api_key = os.environ.get("ANTHROPIC_API_KEY")

print("model:", MODEL)
print("API key loaded:", bool(api_key))

## Slide 11 &middot; Popular AI Packages: langgraph

In [ ]:
def greet_node(state):
    return {"message": f"Hello, {state['name']}!"}


def run_graph(nodes, state):
    for node in nodes:
        state.update(node(state))
    return state


result = run_graph([greet_node], {"name": "Alice"})
print(result["message"])

## Slide 14 &middot; Building a Tiny Agent with LangGraph

In [ ]:
def get_weather(city):
    # pretend this calls a real weather API
    return f"It is sunny in {city}."


def agent_node(state):
    question = state["question"]
    if "weather" in question.lower():
        answer = get_weather("Boston")
    else:
        answer = "I can only answer weather questions."
    return {"answer": answer}


print(agent_node({"question": "What is the weather like?"})["answer"])
print(agent_node({"question": "Who won the game?"})["answer"])

---

### Your turn 2

Extend the tiny agent with a second condition so it also answers a simple addition question, and still declines everything else.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
def agent_node(state):
    question = state["question"].lower()

    if "weather" in question:
        answer = get_weather("Boston")
    # TODO: add a branch that catches a maths question, then a catch-all.
    ____ "plus" ____ question:
        answer = "That is 4."
    ____:
        answer = "I can only answer weather or simple maths right now."

    return {"answer": answer}


for q in ["What is the weather like?", "what is 2 plus 2", "Who won the game?"]:
    print(q, "->", agent_node({"question": q})["answer"])

---

# More use cases

The same ideas, applied to situations you will meet in real work. Run each one, then change a value and run it again.

## Use case 1 &middot; AI &middot; Read your whole config in one block

In [ ]:
import os

config = {
    "model": os.environ.get("LAB_MODEL", "gpt-4o-mini"),
    "temperature": float(os.environ.get("LAB_TEMPERATURE", "0.7")),
    "max_tokens": int(os.environ.get("LAB_MAX_TOKENS", "512")),
    "region": os.environ.get("AWS_REGION", "us-east-1"),
}

for key, value in config.items():
    print(f"  {key:12s} {value}  {type(value).__name__}")

print()
print("API key set:", bool(os.environ.get("LAB_API_KEY")))

## Use case 2 &middot; AI &middot; What did this run cost, and how long did it take?

In [ ]:
import time

start = time.time()

# stands in for three model calls
time.sleep(0.3)

input_tokens = 4_200
output_tokens = 850
elapsed = time.time() - start

cost = (input_tokens / 1_000_000) * 3.00 + (output_tokens / 1_000_000) * 15.00

print("=== run summary ===")
print("input tokens :", input_tokens)
print("output tokens:", output_tokens)
print(f"wall clock   : {elapsed:.2f}s")
print(f"cost         : ${cost:.4f}")
print(f"cost per 1000 runs: ${cost * 1000:.2f}")

## Use case 3 &middot; AI &middot; Summarise a set of latencies

In [ ]:
import statistics

latencies_ms = [45, 52, 38, 512, 47, 41, 60, 44]

print("count  :", len(latencies_ms))
print("mean   :", round(statistics.mean(latencies_ms), 1))
print("median :", statistics.median(latencies_ms))
print("slowest:", max(latencies_ms))
print("fastest:", min(latencies_ms))

## Use case 4 &middot; A timestamped artefact name

In [ ]:
import datetime as dt

now = dt.datetime(2026, 8, 19, 14, 30, 5)      # fixed so the output is stable

stamp = now.strftime("%Y%m%d-%H%M%S")
report = f"eval-run-{stamp}.jsonl"

print("stamp :", stamp)
print("report:", report)
print("today would be:", dt.date.today())

---

## Lab: A timed, config-driven health reporter

Write a small script in this notebook that does four things.

Read a model name from an environment variable called `LAB_MODEL`, falling back
to `"gpt-4o-mini"` when it is not set. Print the model, and separately print
whether an API key called `LAB_API_KEY` is present, without ever printing its
value.

Use `os.listdir` to count how many files sit in the current folder.

Time a fake `run_healthcheck()` function that sleeps briefly and returns a
status, and print the elapsed time to two decimal places.

Stamp the report with today's date using `datetime`.

Print all of it as one tidy report block.

**Done when:**

- [ ] The model comes from the environment with a fallback
- [ ] The key is reported as present or absent, never printed
- [ ] os is used to count the files in the folder
- [ ] time measures the health check, datetime stamps the report

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

The four exercises from the module's practice slide are in the
[README](../README.md#practice-exercises) and repeated on the slide. There are
4 of them. Do them in a scratch cell here or in a `.py` file.

## Module complete

You can now import modules, manage secrets, and build with real AI packages. Exercise 2 needs `python-dotenv`, which is in `requirements-ai.txt`.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*